In [ ]:
import os
import json

from dotenv import load_dotenv

from langchain_community.document_loaders import PDFPlumberLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

from langchain_core.prompts import PromptTemplate
from langchain.chains.llm import LLMChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import RetrievalQA

load_dotenv()

In [14]:
def load_all_documents(pdf_folder):
    """
    Loads all PDF documents from the specified folder using PDFPlumberLoader.
    Returns a list of document objects.
    """
    all_docs = []

    if not os.path.exists(pdf_folder):
        print(f"PDF folder '{pdf_folder}' not found. Make sure your PDFs are in the folder.")
        return all_docs

    for filename in os.listdir(pdf_folder):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, filename)
            print("Loading:", pdf_path)

            loader = PDFPlumberLoader(pdf_path)
            all_docs.extend(loader.load())

    return all_docs

In [15]:
def get_vector_store(documents, embedder, index_dir):
    """
    Creates or loads a FAISS vector store from the provided documents
    using the specified embedder.
    """

    if os.path.exists(index_dir):
        vector = FAISS.load_local(
            index_dir,
            embedder,
            allow_dangerous_deserialization=True
        )
        print("Loaded vector store from disk.")

    else:
        # Split documents into chunks
        text_splitter = SemanticChunker(
            HuggingFaceEmbeddings()
        )

        docs_chunks = text_splitter.split_documents(documents)

        # Create FAISS vector store
        vector = FAISS.from_documents(
            docs_chunks,
            embedder
        )

        # Save vector store locally
        vector.save_local(index_dir)

        print("Created and saved new vector store.")

    return vector

In [16]:
def build_llm_chain(llm):
    """
    Creates an LLMChain using the provided LLM and a QA prompt.
    """
    prompt = """
1. Use the following pieces of context to answer the question at the end.
2. If you don't know the answer, just say "I don't know" and don't make up an answer.
3. Keep the answer crisp and limited to 3-4 sentences.

Context: {context}

Question: {question}

Helpful Answer:
"""

    QA_CHAIN_PROMPT = PromptTemplate.from_template(prompt)

    llm_chain = LLMChain(
        llm=llm,
        prompt=QA_CHAIN_PROMPT,
        verbose=True
    )

    return llm_chain


def build_combined_documents_chain(llm_chain):
    """
    Creates a StuffDocumentsChain for combining retrieved documents.
    """

    document_prompt = PromptTemplate(
        input_variables=["page_content", "source"],
        template="""
Context:
content: {page_content}
source: {source}
"""
    )

    combined_chain = StuffDocumentsChain(
        llm_chain=llm_chain,
        document_variable_name="context",
        document_prompt=document_prompt,
    )

    return combined_chain

In [17]:
def build_qa_chain(retriever, combined_documents_chain):
    """
    Builds and returns a RetrievalQA chain using the provided retriever and combined documents chain.
    """
    qa_chain = RetrievalQA(
        combine_documents_chain=combined_documents_chain,
        retriever=retriever,
        return_source_documents=True,
        verbose=True,
    )
    return qa_chain

In [18]:
load_dotenv()

False

In [19]:
def initialize_chain():
    """
    Initializes the full RetrievalQA chain.
    """

    pdf_folder = os.path.join("static", "uploads")

    # Load PDF documents
    all_docs = load_all_documents(pdf_folder)

    if not all_docs:
        print("No PDF documents loaded.")
        return None

    # Initialize embeddings and vector store
    index_dir = "faiss_index"

    embedder = HuggingFaceEmbeddings()

    vector = get_vector_store(
        all_docs,
        embedder,
        index_dir
    )

    # Create retriever
    retriever = vector.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 1}
    )

    # Fetch Groq API key from .env
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError(
            "GROQ_API_KEY not found. Please add it to your .env file."
        )

    # Initialize Groq LLM
    llm = ChatGroq(
        groq_api_key=GROQ_API_KEY,
        model_name="openai/gpt-oss-20b",
        temperature=0.01,
        max_retries=2
    )

    # Build LLM chain
    llm_chain = build_llm_chain(llm)

    # Combine retrieved documents
    combined_documents_chain = build_combined_documents_chain(
        llm_chain
    )

    # Build RetrievalQA chain
    qa_chain = build_qa_chain(
        retriever,
        combined_documents_chain
    )

    return qa_chain

In [30]:
def get_response(qa_chain, question):
    """
    Runs the QA chain with the provided question.
    Returns the answer and a relative URL for the source PDF (if available).
    """

    response = qa_chain.invoke({
        "query": question
    })

    answer_text = response.get("result", "I don't know")
    pdf_url = None

    # Get source document
    if response.get("source_documents"):

        doc = response["source_documents"][0]
        metadata = doc.metadata

        source_doc = metadata.get("source", "")
        page_num = metadata.get("page", 0)

        # Normalize path
        normalized_source = source_doc.replace("\\", "/")

        # Remove leading "static/"
        if normalized_source.lower().startswith("static/"):
            normalized_source = normalized_source[len("static/"):]

        # Create PDF URL with page number
        pdf_url = f"/static/{normalized_source}#page={page_num + 1}"

    return answer_text, pdf_url

In [ ]:
qa_chain = initialize_chain()

In [ ]:
qa_chain = initialize_chain()

print(qa_chain)

In [ ]:
pdf_folder = os.path.join("static", "uploads")

all_docs = load_all_documents(pdf_folder)

print("Number of documents loaded:", len(all_docs))

In [35]:
sample_question = "What are the essential requirements regarding the composition and the reusable and recoverable, including recyclable, nature of packaging?"

In [ ]:
answer, pdf_url = get_response(qa_chain, sample_question)

print("Question:", sample_question)
print("Answer:", answer)
print("PDF URL:", pdf_url)